In [ ]:
!pip install gymnasium minigrid imageio

In [ ]:
import gymnasium as gym
import minigrid
import imageio

# 1. Crie o ambiente com o modo de renderização em RGB
env = gym.make("MiniGrid-Empty-8x8-v0", render_mode="rgb_array")
obs, info = env.reset()

frames = []

# 2. Rode o ambiente e salve os frames
for _ in range(20): # 20 passos de exemplo
    frame = env.render()
    frames.append(frame)

    action = env.action_space.sample() # Escolha uma ação (ex: aleatória)
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs, info = env.reset()

env.close()

# 3. Salve os frames como um GIF animado
imageio.mimsave('meu_grid_animado.gif', frames, fps=10) # fps controla a velocidade

In [ ]:
!pip install minigrid==3.0.0

In [ ]:
import time
import os
import re
from google.colab import userdata

import gymnasium as gym
import minigrid

from google import genai

In [ ]:
API_KEY = userdata.get('GOOGLE_PESSOAL')

GEMINI_CLIENT = genai.Client(api_key=API_KEY)
GEMINI_MODEL_ID = "gemini-2.5-flash"

In [ ]:
MAP_CELLS = {'wall': '#', 'floor': '.', 'goal': 'O', 'key': 'C', 'lava': 'L', 'open_door': '_', 'locked_dor': 'X', 'unlocked_closed_door': 'P'}
PLAYER_DIRECTIONS = ['>', 'v', '<', '^']

class MiniGridTextWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.map_cells_repr = MAP_CELLS
        self.directions_repr = PLAYER_DIRECTIONS
        self.unknown = "?"

    def observation(self, obs):
        env = self.unwrapped
        agent_pos = env.agent_pos
        current_room = None
        if hasattr(env, 'rooms'):
            for room in env.rooms:
                top_x, top_y = room.top
                w, h = room.size
                if (top_x <= agent_pos[0] < top_x + w and top_y <= agent_pos[1] < top_y + h):
                    current_room = room
                    break

        x_s, y_s = (current_room.top, current_room.size) if current_room else ((0,0), (env.grid.width, env.grid.height))

        # Grid logic for the loop execution:
        grid_str = ""
        last_line = y_s[1] - 1
        for y in range(y_s[1]):
            line = ""
            for x in range(x_s[0], x_s[0] + y_s[0]):
                real_x, real_y = x, y + x_s[1]
                if [real_x, real_y] == list(agent_pos):
                    line += self.directions_repr[env.agent_dir]
                else:
                    tile = env.grid.get(real_x, real_y)
                    if tile is None:
                        line += self.map_cells_repr['floor']
                    elif tile.type == 'door':
                        line += (MAP_CELLS['open_door'] if tile.is_open else (MAP_CELLS['locked_dor'] if tile.is_locked else MAP_CELLS['unlocked_closed_door']))
                    else:
                        line += self.map_cells_repr.get(tile.type, self.unknown)
            grid_str += line + ("\n" if y != last_line else "")

        return grid_str

In [ ]:
# Função para facilitar a geração de respostas com o modelo
def generate_gemini_response(system_prompt, prompt):
    response = GEMINI_CLIENT.models.generate_content(
        model=GEMINI_MODEL_ID,
        contents=prompt,
        config={'system_instruction': system_prompt, 'temperature': 0.1}
    )
    return response.text

In [ ]:
original_env = gym.make("MiniGrid-LavaCrossingS9N3-v0")
# O ambiente original é encapsulado em um wrapper que converte as observações para texto
# Obs.: Aqui, usamos a representação textual "grande" porque é melhor para visualização humana
env = MiniGridTextWrapper(original_env)

obs, _ = env.reset()

print("Observação inicial (mapa):")
print(obs)
# loop aplicando sequência de 5 ações aleatórias
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    print("-- ação (aleatória) --")
    print(action)

    print("-- estado --")
    print(obs)

    #print("-- recompensa --")
    #print(reward)

    if terminated or truncated:
        break

In [ ]:
SYSTEM_PROMPT = """
Você controla um agente que navega em um ambiente quadriculado 2D (um "grid").
Seu objetivo é guiar o agente pelo ambiente para encontrar e alcançar a posição do objetivo.

# REGRAS DO AMBIENTE:
- Você receberá a observação da sua sala atual, na forma de um "grid" representado em texto.
- Cada célula da sala é representada por um único caractere, seguindo esta representação:
    - '#' (parede)
    - '.' (chão, célula vazia)
    - 'O' (posição do objetivo)
    - 'L' (lava)
- Indicadores do agente: serão colocados na representação do ambiente para mostrar sua posição atual e a direção para a qual você está voltado.
  - Pode ser um dos seguintes: '^' (cima), 'v' (baixo), '<' (esquerda) ou '>' (direita).
- Ações disponíveis:
    - GIRA_ANTI_HORARIO: Gira 90 graus no sentido anti-horário.
    - GIRA_HORARIO: Gira 90 graus no sentido horário.
    - FRENTE: Move um passo na direção para a qual você está voltado.

# FORMATO DA RESPOSTA:
Sua resposta deve ter apenas duas linhas e NADA MAIS. Não inclua texto conversacional.
THOUGHT: <Explique um breve raciocínio para justificar a escolha da próxima ação>
ACTION: <Uma das ações listadas acima>

## Exemplo de resposta:
THOUGHT: Estou de frente para o objetivo, que está na próxima célula adiante.
ACTION: FRENTE
"""

In [ ]:
observation_template = """# OBSERVAÇÃO ATUAL:
{SALA_ATUAL}
"""

In [ ]:
# Para mapear as ações textuais em códigos numéricos aceitos pelo ambiente gym
ACTION_LOOKUP = {
    "GIRA_ANTI_HORARIO": 0,
    "GIRA_HORARIO": 1,
    "FRENTE": 2,
}

In [ ]:
def extract_thought_and_action(response_text):
    thought_match = re.search(r"THOUGHT: (.*)", response_text)
    thought_str = thought_match.group(1) if thought_match else "(not found)"

    action_match = re.search(r"ACTION: (GIRA_ANTI_HORARIO|GIRA_HORARIO|FRENTE)", response_text)
    if action_match:
        action_str = action_match.group(1)
    else:
        print(f"Error: Model failed to format action correctly. Response: {response_text}")
        action_str = ""

    return thought_str, action_str

In [ ]:
env = MiniGridTextWrapper(original_env)
# Reinicia o ambiente
obs, _ = env.reset()
terminated = False
truncated = False
step_count = 0


In [ ]:
# Loop Principal
print("Running ReAct Agent loop...\n")

print("-" * 20)
print("OBSERVAÇÃO INICIAL:")
print(obs)
print("-" * 20)

while not (terminated or truncated) and step_count < 20:
    # 1. Prepare the prompt with current observation
    obs_prompt = observation_template.format(SALA_ATUAL=obs)

    # 2. Get Response from Language Model
    response_text = generate_gemini_response(SYSTEM_PROMPT, obs_prompt)

    # 3. Parse Thought and Action from the string response
    thought, action_str = extract_thought_and_action(response_text)

    # 4. Display the ReAct cycle
    print("-" * 20)
    print(f"Passo: {step_count}")
    print("-" * 20)
    print("PENSAMENTO DO MODELO:", thought)
    print("ACAO ESCOLHIDA:", action_str)
    #print()

    # 5. Execute action
    action_number = ACTION_LOOKUP[action_str]
    obs, reward, terminated, truncated, _ = env.step(action_number)
    step_count += 1

    # 6. Print the new resulting observation
    print("NOVA OBSERVAÇÃO:")
    print(obs)
    print("-" * 20)

    # 7. Pause slightly so we can watch the agent work
    time.sleep(3.0)

if terminated:
    if reward > 0:
        print("\nSUCCESS: Model reached the goal!")
    else:
        print("\nFAILURE: The agent died or failed!")

env.close()
print("Environment closed.")